In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../retailiq.db")

payments = pd.read_sql("SELECT * FROM payments_clean", conn)
print(payments["payment_type"].value_counts())

# Treatment: did this order involve a voucher payment (proxy for discount usage)?
voucher_orders = payments[payments["payment_type"] == "voucher"]["order_id"].unique()
print(f"\nOrders with voucher payment: {len(voucher_orders)} out of {payments['order_id'].nunique()} total orders")

payment_type
credit_card    76795
boleto         19784
voucher         5769
debit_card      1529
Name: count, dtype: int64

Orders with voucher payment: 3866 out of 99437 total orders


In [2]:
order_to_customer = pd.read_sql(
    "SELECT o.order_id, c.customer_unique_id FROM orders_clean o "
    "JOIN customers_clean c ON o.customer_id = c.customer_id", conn)

voucher_df = pd.DataFrame({"order_id": voucher_orders})
voucher_df["used_voucher"] = 1
voucher_by_customer = order_to_customer.merge(voucher_df, on="order_id", how="left")
voucher_by_customer["used_voucher"] = voucher_by_customer["used_voucher"].fillna(0)

treatment = voucher_by_customer.groupby("customer_unique_id")["used_voucher"].max().reset_index()
treatment.columns = ["customer_unique_id", "treatment_voucher"]

print(treatment["treatment_voucher"].value_counts())
print(f"\n% of customers who ever used a voucher: {round(100*treatment['treatment_voucher'].mean(), 1)}%")

# Merge into the main feature table
features = pd.read_sql("SELECT * FROM customer_features", conn)
features = features.merge(treatment, on="customer_unique_id", how="left")
features["treatment_voucher"] = features["treatment_voucher"].fillna(0)

features.to_sql("customer_features", conn, if_exists="replace", index=False)
conn.close()
print("\nUpdated customer_features table with treatment_voucher column")

treatment_voucher
0.0    91510
1.0     3723
Name: count, dtype: int64

% of customers who ever used a voucher: 3.9%

Updated customer_features table with treatment_voucher column
